# Dwell-Time Simulator
**BINARY DISTRIBUTION · CONFOUNDING ANALYSIS**

---


## Inhalt

  - [Kurz erklärt: Was ist Dwell-Time-Konfundierung?](#kurz-erklärt-was-ist-dwell-time-konfundierung)
- [Setup](#setup)
- [Schritt 1 — dwell_time Verteilung](#schritt-1-dwelltime-verteilung)
- [Schritt 2 — Konfundierungs-Analyse: dwell_time × Delay](#schritt-2-konfundierungs-analyse-dwelltime-delay)
- [Schritt 3 — Simulation: 0 → 60s (einziger valider Wert)](#schritt-3-simulation-0-60s-einziger-valider-wert)
- [Schritt 4 — Netzweit: 0 → 60s](#schritt-4-netzweit-0-60s)
- [Key Findings](#key-findings)


**Fragestellung:** Was wäre wenn VBZ an Haltestellen mehr Puffer einplant?

`dwell_time` ist das **#1-Feature** in LightGBM v1 (Gain: 14.8M — vor `stop_name` 12.7M).
Gleichzeitig zeigt F-SPAT-08: 71.3% aller Halte haben `dwell_time = 0`. Der stärkste
Modell-Prädiktor ist gleichzeitig die grösste strukturelle Lücke im Fahrplan.

Dieses Notebook führt die Simulation durch und stellt dabei eine grundsätzliche Frage:
**Kann ein auf Beobachtungsdaten trainiertes Modell kausale Empfehlungen liefern?**

**Warum LightGBM v1 (nicht v2)?**
v2 hat `prev_trip_delay` — ein Kaskadenfeature, das sich selbst verändern würde,
wenn dwell_time sich ändert. v1 ist self-contained: eine Änderung in `dwell_time`
→ direkt eine neue Vorhersage, ohne Kaskadenabhängigkeit.

**Schließt den Kreis:**
Analyse (F-SPAT-08: kein Puffer) → Modell (Feature #1) → Simulation → Befund über ML-Grenzen

### Kurz erklärt: Was ist Dwell-Time-Konfundierung?

**Für alle, die das Notebook ohne Vorkenntnisse lesen:**

`dwell_time` ist die geplante Aufenthaltsdauer einer Tram an einer Haltestelle (0s oder 60s).
Es ist das Feature mit dem höchsten Gain-Score im Modell — noch vor `stop_name`.

**Die Konfundierung:** Haltestellen mit `dwell_time = 60s` haben im Durchschnitt auch mehr Delay
(r = +0.16). Nicht weil der Puffer Delays verursacht — sondern weil die VBZ an *komplizierten*
Haltestellen (viel Fahrgastwechsel, Umsteigeknoten) Puffer einplant, und genau diese Haltestellen
sind strukturell anfälliger für Delays.

```
Komplizierter Stop (z.B. Stauffacher, Leutschenbach)
         ↓                              ↓
  VBZ gibt 60s dwell_time         Hohe Fahrgastzahl
  (um Anschlüsse zu sichern)      → mehr Delay
```

Das Modell hat diese Korrelation korrekt gelernt. Das Problem: wenn wir für die Simulation
einem *einfachen* Stop +60s geben, interpretiert das Modell das als "dieser Stop ist jetzt
ein komplizierter Umsteigeknoten" — und erhöht die Delay-Vorhersage entsprechend.

**Feature Importance ≠ kausaler Hebel.** Das ist die Kernaussage dieses Notebooks.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import json
from pathlib import Path

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("06_prediction_6-dwell_simulator")

MODELS_DIR = Path(TRAIN).parent.parent / "models"
%load_ext autoreload
%autoreload 2

In [ ]:
model = lgb.Booster(model_file=str(MODELS_DIR / "lgbm_v1.txt"))

with open(MODELS_DIR / "lgbm_v1_meta.json") as f:
    meta = json.load(f)

FEATURE_COLS = meta["features"]
CAT_COLS = meta["cat_cols"]

print(f"Features: {len(FEATURE_COLS)} | #1 by gain: dwell_time")
print(f"Cat cols: {CAT_COLS}")

In [ ]:
# test_final.parquet hat alle Modell-Features (gtfs_year, dwell_time, n_lines_at_stop, etc.)
# test_features.parquet ist die Zwischen-Stufe vor dem Feature-Engineering → hier NICHT verwenden
TEST_FINAL = PATHS["processed"] / "test_final.parquet"

# Polars Lernmoment: .cast(pl.Utf8) stellt sicher dass line_name immer String ist,
# unabhängig davon ob es im Parquet als Int oder Str gespeichert ist.
df_test = (
    pl.scan_parquet(TEST_FINAL)
    .with_columns(pl.col("line_name").cast(pl.Utf8))
    .collect()
    .to_pandas()
)
for col in CAT_COLS:
    df_test[col] = df_test[col].astype("category")

print(f"Test rows: {len(df_test):,}")
print(f"Columns: {len(df_test.columns)} (model needs {len(FEATURE_COLS)})")

## Schritt 1 — dwell_time Verteilung

Bevor wir simulieren: welche Werte nimmt `dwell_time` überhaupt an?

In [ ]:
dwell_vc = df_test["dwell_time"].value_counts().sort_index()
print("dwell_time Werteverteilung (Test-Set):")
print(dwell_vc[dwell_vc > 100].to_string())
print()
print(f"Anteil dwell_time = 0:  {(df_test['dwell_time'] == 0).mean():.1%}")
print(f"Anteil dwell_time = 60: {(df_test['dwell_time'] == 60).mean():.1%}")
print(f"Anteil dwell_time 1–59: {((df_test['dwell_time'] > 0) & (df_test['dwell_time'] < 60)).mean():.1%}")

**Befund:** `dwell_time` ist in VBZ-Fahrplandaten **faktisch binär**: entweder `0s` (71.3%) oder `60s` (28.5%).
Werte zwischen 1 und 59 Sekunden existieren nicht — die VBZ plant entweder keinen Puffer (0) oder einen
vollen Aufenthalt (60s). Das hat direkte Konsequenzen für die Simulation:
Eine Simulation mit +10s würde Werte erzeugen, die das Modell **noch nie gesehen hat** —
out-of-distribution Extrapolation.

## Schritt 2 — Konfundierungs-Analyse: dwell_time × Delay

Korreliert höhere dwell_time mit weniger oder mehr Verspätung?

In [ ]:
r = df_test["dwell_time"].corr(df_test["arrival_delay"])
print(f"Pearson r (dwell_time × arrival_delay) = {r:.4f}")
print()

# Mean delay by dwell_time value (binary comparison)
summary = (
    df_test.groupby("dwell_time")["arrival_delay"]
    .agg(["mean", "count"])
    .loc[[0, 60]]
)
summary.columns = ["Ø arrival_delay (s)", "N"]
summary.index.name = "dwell_time"
show_df(summary)

**Befund:** Pearson r = +0.16 — `dwell_time` korreliert **positiv** mit Verspätung.
Haltestellen mit `dwell_time = 60s` haben im Durchschnitt ~28s mehr Delay als
Haltestellen mit `dwell_time = 0`.

**Erklärung — Konfundierung durch Haltestellen-Komplexität:**

```
Komplexer Stop
    ↓              ↓
dwell_time = 60   arrival_delay hoch
(VBZ plant Puffer  (viel Betrieb,
 an schwierigen    MIV-Interaktion,
 Stops ein)        Fahrgastwechsel)
```

VBZ gibt mehr `dwell_time` an Umsteigepunkten (Stauffacher, Bahnhof Oerlikon, Leutschenbach) —
genau dort, wo strukturell mehr Delay entsteht. Das Modell hat diese Korrelation korrekt gelernt.
Aber: **Correlation ≠ Causation.** Wenn wir einem einfachen Halt +60s geben, wird er
dadurch nicht zu Stauffacher — der Modell-Prädiktor ist hier ein **Proxy**, kein Hebel.

## Schritt 3 — Simulation: 0 → 60s (einziger valider Wert)

Simulation mit dem einzigen Wert, den das Modell trainiert hat: `dwell_time = 0 → 60`.
Was sagt das Modell für Haltestellen, die aktuell keinen Puffer haben?

In [ ]:
def simulate_dwell(
    df: pd.DataFrame,
    new_value: int = 60,
    line_name: str | None = None,
) -> pd.DataFrame:
    """Simulate changing dwell_time from 0 to new_value at zero-dwell stops.

    Args:
        df: Test dataframe with pred_base already computed.
        new_value: New dwell_time value (should be a value seen in training — 60 or 120).
        line_name: If given, restrict to this line. None = network-wide.

    Returns:
        Copy of df with pred_sim and delta_delay columns added.
    """
    df_sim = df.copy()
    mask = df_sim["dwell_time"] == 0
    if line_name is not None:
        mask = mask & (df_sim["line_name"].astype(str) == str(line_name))
    df_sim.loc[mask, "dwell_time"] = new_value
    df_sim["pred_sim"] = model.predict(df_sim[FEATURE_COLS])
    df_sim["delta_delay"] = df_sim["pred_sim"] - df_sim["pred_base"]
    if line_name is not None:
        return df_sim[df_sim["line_name"].astype(str) == str(line_name)].copy()
    return df_sim


# Baseline first
df_test["pred_base"] = model.predict(df_test[FEATURE_COLS])
mae_base = np.abs(df_test["pred_base"] - df_test["arrival_delay"]).mean()
print(f"Baseline MAE: {mae_base:.2f}s")

In [ ]:
# L11: 0 → 60s
sim_l11 = simulate_dwell(df_test, new_value=60, line_name="11")

mae_l11_base = np.abs(sim_l11["pred_base"] - sim_l11["arrival_delay"]).mean()
mae_l11_sim  = np.abs(sim_l11["pred_sim"]  - sim_l11["arrival_delay"]).mean()
mean_delta   = sim_l11["delta_delay"].mean()

print(f"L11 — 0→60s dwell_time:")
print(f"  MAE Baseline:  {mae_l11_base:.2f}s")
print(f"  MAE Simulation:{mae_l11_sim:.2f}s")
print(f"  Ø Δ Delay:     {mean_delta:+.2f}s")

In [ ]:
# Per-stop breakdown
stops = (
    sim_l11.groupby("stop_name", observed=True)
    .agg(
        mean_delta=("delta_delay", "mean"),
        mean_actual=("arrival_delay", "mean"),
        n=("delta_delay", "count"),
    )
    .reset_index()
    .sort_values("mean_actual", ascending=False)
)

stop_labels = stops["stop_name"].str.replace("Zürich, ", "", regex=False)
colors = ["#25ac82" if v <= 0 else "#de425b" for v in stops["mean_delta"]]

fig = go.Figure(go.Bar(
    x=stop_labels,
    y=stops["mean_delta"].round(1),
    marker_color=colors,
    hovertemplate="<b>%{x}</b><br>Δ Delay: %{y:.1f}s<extra></extra>",
))
fig.add_hline(y=0, line_dash="dot", line_color="#888", line_width=1)
fig.update_layout(
    title=dict(
        text=(
            f"L11 — Modell-Vorhersage: Δ Delay bei dwell_time 0→60s<br>"
            f"<sup>Grün = Verbesserung · Rot = Verschlechterung · Ø {mean_delta:+.1f}s</sup>"
        ),
        x=0, xanchor="left",
    ),
    xaxis=dict(title="Haltestelle", tickangle=-40),
    yaxis=dict(title="Δ Delay (s)"),
    height=480,
    margin=dict(l=0, r=0, t=80, b=130),
    plot_bgcolor="white",
)
fig.show()

# Companion table
stops_display = stops.copy()
stops_display.insert(0, "Haltestelle", stops_display["stop_name"].str.replace("Zürich, ", "", regex=False))
stops_display["Ø Delay Ist (s)"] = stops_display["mean_actual"].round(1)
stops_display["Δ Delay 0→60s (s)"] = stops_display["mean_delta"].round(1)
stops_display["N"] = stops_display["n"]
show_df(stops_display[["Haltestelle", "Ø Delay Ist (s)", "Δ Delay 0→60s (s)", "N"]].set_index("Haltestelle"))

## Schritt 4 — Netzweit: 0 → 60s

In [ ]:
sim_net = simulate_dwell(df_test, new_value=60, line_name=None)

mae_net_base = np.abs(sim_net["pred_base"] - sim_net["arrival_delay"]).mean()
mae_net_sim  = np.abs(sim_net["pred_sim"]  - sim_net["arrival_delay"]).mean()
mean_delta_net = sim_net["delta_delay"].mean()

print(f"Netzweit — 0→60s:")
print(f"  MAE Baseline:  {mae_net_base:.2f}s")
print(f"  MAE Simulation:{mae_net_sim:.2f}s")
print(f"  Ø Δ Delay:     {mean_delta_net:+.2f}s")

# Per-line summary
line_summary = (
    sim_net.groupby(sim_net["line_name"].astype(str))
    .agg(
        mean_delta=("delta_delay", "mean"),
        n=("delta_delay", "count"),
    )
    .reset_index()
    .sort_values("mean_delta")
)
line_summary["Ø Δ Delay (s)"] = line_summary["mean_delta"].round(1)
line_summary.insert(0, "Linie", line_summary["line_name"].apply(lambda x: f"L{x}"))
show_df(line_summary[["Linie", "Ø Δ Delay (s)", "n"]].set_index("Linie"))

## Key Findings

→ Vollständige Findings-Tabelle in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

`Präsentation`: **hot** = Kernbefund · **story** = gutes Narrativ · **—** = intern

| ID | Finding | Präsentation |
|:---|:---|:---:|
| F-SIM-01 | `dwell_time` ist Feature #1 in v1 (Gain 14.8M), aber faktisch **binär** in VBZ-Daten: entweder 0s (71.3%) oder 60s (28.5%). Werte zwischen 1–59s existieren nicht — jede Simulation mit +10s, +20s wäre out-of-distribution. | **hot** |
| F-SIM-02 | `dwell_time` korreliert **positiv** mit Delay (r=+0.16): Stops mit 60s haben ~28s mehr Delay als Stops mit 0s. Ursache: Konfundierung durch Stopschwierigkeit — VBZ gibt Puffer an komplexen Stops, die auch mehr Delay haben. Das Modell hat die Korrelation korrekt gelernt. | **hot** |
| F-SIM-03 | Simulation 0→60s: Modell erhöht Delay-Vorhersage um **+20s** (L11: +19.96s, Netzweit: +20.72s) — keine Verbesserung. **Feature Importance ≠ kausaler Hebel.** Das Modell kann nicht unterscheiden ob ein Stop dwell_time=60 hat weil er komplex ist (historisch) oder weil VBZ ihm Puffer gibt (hypothetisch). | **story** |
| F-SIM-04 | Operative Empfehlung trotzdem valide (Domänenwissen + F-SPAT-08): stopspezifische dwell_time kalibrieren statt pauschaler 0/60s — manche Stops brauchen 15s, andere 90s. Quantifizierung des Nutzens erfordert A/B-Test oder Instrumental Variable. Observational ML kann den Kausaleffekt nicht isolieren. | **story** |